# RGCA Research-Safe Kaggle Baseline

Run this notebook from top to bottom.

It handles the full baseline workflow:

1. Clone/update the RGCA repository.
2. Install the repo package.
3. Detect an attached private Kaggle dataset containing `mimic_subset.jsonl`.
4. If no subset is attached, prepare a pilot subset from PhysioNet using Kaggle Secrets.
5. Validate the subset and run the guarded baseline suite.
6. Inspect result tables and sample generations.
7. Package outputs for reuse as a private Kaggle dataset.

This notebook does **not** use GCloud.

Execution modes:

- `debug`: code smoke tests only; demo data allowed only when explicitly enabled.
- `stress`: current baseline mode; real MIMIC-derived subset required; controlled retrieval-copy stress tests allowed.
- `real`: reserved for real VLM/image retrieval; currently fails by design until those backends exist.


## 1. Repository Setup

This cell makes sure Kaggle is using the latest `pidoxy/RGCA` code.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/kaggle/working/RGCA')
REPO_URL = 'https://github.com/pidoxy/RGCA.git'

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', 'main'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)

print('Repo ready:', PROJECT_ROOT)
print('Python:', sys.version)


## 2. Experiment Configuration

For the current baseline, keep `EXECUTION_MODE = "stress"`.

If you already attached `rgca_private_dataset`, the notebook will use it automatically.

If no private dataset is attached, the notebook will prepare the subset from PhysioNet. For that path, create and enable these Kaggle Secrets:

- `PHYSIONET_USERNAME`
- `PHYSIONET_PASS`


In [ ]:
EXECUTION_MODE = 'stress'  # debug | stress | real
ALLOW_DEMO_DEBUG = False  # Set True only for code smoke tests, never for research evidence.

WORK_DIR = Path('/kaggle/working/physionet')
PILOT_OUTPUT_DIR = Path('/kaggle/working/rgca_pilot_500')
SUITE_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0')
PRIVATE_DATASET_ZIP = Path('/kaggle/working/rgca_private_dataset.zip')

RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100


def read_kaggle_secret(name, default=''):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or default
    except Exception:
        return default

PHYSIONET_USERNAME = (
    os.environ.get('PHYSIONET_USERNAME')
    or read_kaggle_secret('PHYSIONET_USERNAME')
    or read_kaggle_secret('PHYSIONET_USER')
)
PHYSIONET_PASS = (
    os.environ.get('PHYSIONET_PASS')
    or read_kaggle_secret('PHYSIONET_PASS')
    or read_kaggle_secret('PHYSIONET_PASSWORD')
)

if PHYSIONET_PASS:
    os.environ['PHYSIONET_PASS'] = PHYSIONET_PASS

print('EXECUTION_MODE:', EXECUTION_MODE)
print('ALLOW_DEMO_DEBUG:', ALLOW_DEMO_DEBUG)
print('PHYSIONET_USERNAME set:', bool(PHYSIONET_USERNAME))
print('PHYSIONET_PASS set:', bool(PHYSIONET_PASS))


## 3. Resolve Or Prepare Dataset

This is the key cell.

It tries, in order:

1. Use an attached Kaggle input containing `mimic_subset.jsonl`.
2. Extract an attached/private zip if one contains `mimic_subset.jsonl`.
3. Use an already prepared working subset at `/kaggle/working/rgca_pilot_500/data/mimic_subset.jsonl`.
4. Prepare a new subset from PhysioNet metadata + reports.
5. Use demo data only if `EXECUTION_MODE='debug'` and `ALLOW_DEMO_DEBUG=True`.


In [ ]:
def find_subset_jsonl(roots):
    matches = []
    for root in roots:
        root = Path(root)
        if root.exists():
            matches.extend(root.rglob('mimic_subset.jsonl'))
    return sorted(set(matches))


def extract_candidate_zips(input_root=Path('/kaggle/input'), extract_root=Path('/kaggle/working/rgca_extracted_inputs')):
    extracted_roots = []
    if not input_root.exists():
        return extracted_roots
    for zip_path in sorted(input_root.rglob('*.zip')):
        target = extract_root / zip_path.stem
        if not target.exists():
            target.mkdir(parents=True, exist_ok=True)
            print('Extracting attached zip:', zip_path, '->', target)
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(target)
        extracted_roots.append(target)
    return extracted_roots


def infer_pilot_output_dir(subset_path):
    subset_path = Path(subset_path)
    if subset_path.parent.name == 'data':
        return subset_path.parent.parent
    return subset_path.parent

input_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
print('Initial /kaggle/input entries:')
for path in sorted(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []:
    print('-', path)

subset_matches = find_subset_jsonl(input_roots)
if not subset_matches:
    extracted_roots = extract_candidate_zips()
    subset_matches = find_subset_jsonl(extracted_roots + input_roots)

if subset_matches:
    SUBSET_JSONL = subset_matches[0]
    PILOT_OUTPUT_DIR = infer_pilot_output_dir(SUBSET_JSONL)
    print('Using existing subset:', SUBSET_JSONL)
else:
    prepared_subset = PILOT_OUTPUT_DIR / 'data' / 'mimic_subset.jsonl'
    if prepared_subset.exists():
        SUBSET_JSONL = prepared_subset
        print('Using prepared working subset:', SUBSET_JSONL)
    elif EXECUTION_MODE == 'debug' and ALLOW_DEMO_DEBUG:
        SUBSET_JSONL = PROJECT_ROOT / 'data' / 'demo' / 'demo_studies.jsonl'
        PILOT_OUTPUT_DIR = SUBSET_JSONL.parent
        print('Using demo subset for debug only:', SUBSET_JSONL)
    else:
        if not PHYSIONET_USERNAME or not PHYSIONET_PASS:
            raise RuntimeError(
                'No attached mimic_subset.jsonl was found and PhysioNet secrets are missing. '
                'Attach rgca_private_dataset, or add and enable Kaggle Secrets PHYSIONET_USERNAME and PHYSIONET_PASS.'
            )
        command = [
            sys.executable,
            'scripts/kaggle_prepare_mimic_subset.py',
            '--physionet-user', PHYSIONET_USERNAME,
            '--work-dir', str(WORK_DIR),
            '--output-dir', str(PILOT_OUTPUT_DIR),
            '--retrieval-limit', str(RETRIEVAL_LIMIT),
            '--eval-limit', str(EVAL_LIMIT),
        ]
        safe_command = [part if part != PHYSIONET_USERNAME else '<PHYSIONET_USERNAME>' for part in command]
        print('+', ' '.join(safe_command))
        subprocess.run(command, check=True, env=os.environ.copy())
        SUBSET_JSONL = prepared_subset

SUBSET_JSONL = Path(SUBSET_JSONL)
PILOT_OUTPUT_DIR = Path(PILOT_OUTPUT_DIR)

if not SUBSET_JSONL.exists():
    raise FileNotFoundError(f'Subset was expected but not found: {SUBSET_JSONL}')

print('SUBSET_JSONL:', SUBSET_JSONL)
print('PILOT_OUTPUT_DIR:', PILOT_OUTPUT_DIR)
print('Subset size MB:', round(SUBSET_JSONL.stat().st_size / (1024 * 1024), 3))


## 4. Validate Dataset Contract

This confirms the subset has retrieval-pool records, eval records, required fields, no duplicate study IDs, and a reproducible fingerprint.

In [ ]:
from rgca_baseline.integrity import assert_dataset_allowed, jsonl_fingerprint, validate_study_records
from rgca_baseline.pipeline import load_studies

try:
    assert_dataset_allowed(SUBSET_JSONL, EXECUTION_MODE)
except ValueError as exc:
    raise RuntimeError(str(exc)) from exc

studies = load_studies(SUBSET_JSONL)
validation = validate_study_records(studies)
fingerprint = jsonl_fingerprint(SUBSET_JSONL)

print(json.dumps({'validation': validation, 'fingerprint': fingerprint}, indent=2))
if not validation['valid']:
    raise RuntimeError('Dataset contract validation failed. Fix the subset before running experiments.')


## 5. Run Guarded Baseline Suite

This runs the full current baseline suite. In `stress` mode, demo/mock-only research runs are blocked.

In [ ]:
command = [
    sys.executable,
    'scripts/kaggle_bootstrap_baseline.py',
    '--subset-jsonl', str(SUBSET_JSONL),
    '--execution-mode', EXECUTION_MODE,
    '--output-dir', str(SUITE_OUTPUT_DIR),
    '--pilot-output-dir', str(PILOT_OUTPUT_DIR),
    '--overwrite',
]
if EXECUTION_MODE == 'debug' and ALLOW_DEMO_DEBUG:
    command.append('--allow-demo')

print('+', ' '.join(command))
subprocess.run(command, check=True)


## 6. Inspect Outputs

The table is the quick result view. The manifest is the audit trail.

In [ ]:
summary_path = SUITE_OUTPUT_DIR / 'bootstrap_summary.json'
manifest_path = SUITE_OUTPUT_DIR / 'suite_manifest.json'
table_path = SUITE_OUTPUT_DIR / 'tables' / 'suite_summary.md'

for path in [summary_path, manifest_path, table_path]:
    print(path, 'exists=', path.exists())

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('\nBootstrap summary:')
    print(json.dumps(summary, indent=2)[:4000])

if table_path.exists():
    print('\nSuite summary table:')
    print(table_path.read_text(encoding='utf-8')[:8000])


## 7. Inspect Example Generations

This prints a few mismatch examples so we can manually inspect retrieval contamination.

In [ ]:
def read_jsonl(path, limit=3):
    rows = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            rows.append(json.loads(line))
            if len(rows) >= limit:
                break
    return rows

experiment_dirs = sorted(path for path in SUITE_OUTPUT_DIR.glob('E*') if path.is_dir())
print('Experiments:', [path.name for path in experiment_dirs])

for experiment_dir in experiment_dirs[:2]:
    gen_path = experiment_dir / 'baseline' / 'generations_mismatch.jsonl'
    ret_path = experiment_dir / 'baseline' / 'mismatch_results.jsonl'
    print('\n===', experiment_dir.name, '===')
    print('generations:', gen_path.exists(), 'retrieval:', ret_path.exists())
    if gen_path.exists():
        for row in read_jsonl(gen_path, limit=2):
            print('\nStudy:', row.get('study_id'))
            print('Generated:', row.get('generated_report', '')[:1000])


## 8. Save And Reuse Outputs

After a successful run, this section prepares two reusable artifacts:

```text
/kaggle/working/mimic_pilot_baseline_v0.zip
/kaggle/working/rgca_private_dataset.zip
```

Download `mimic_pilot_baseline_v0.zip` from the Kaggle Output panel when you want the full experiment evidence folder.

For future notebooks, create or update a private Kaggle dataset with `rgca_private_dataset.zip` or its extracted contents, attach it as input, and rerun this notebook. It will automatically find `mimic_subset.jsonl` and skip PhysioNet download.


In [ ]:
EVIDENCE_ZIP = Path('/kaggle/working/mimic_pilot_baseline_v0.zip')

if SUITE_OUTPUT_DIR.exists():
    archive_base = EVIDENCE_ZIP.with_suffix('')
    shutil.make_archive(str(archive_base), 'zip', str(SUITE_OUTPUT_DIR))
    print('Evidence folder zipped for download:', EVIDENCE_ZIP)
    print('Evidence zip size MB:', round(EVIDENCE_ZIP.stat().st_size / (1024 * 1024), 3))
else:
    raise FileNotFoundError(f'Cannot create evidence zip because suite output is missing: {SUITE_OUTPUT_DIR}')

print('\nPrivate dataset zip expected at:', PRIVATE_DATASET_ZIP)
print('Exists:', PRIVATE_DATASET_ZIP.exists())
if PRIVATE_DATASET_ZIP.exists():
    print('Size MB:', round(PRIVATE_DATASET_ZIP.stat().st_size / (1024 * 1024), 3))
else:
    print('Private dataset zip was not found. The evidence zip above is still downloadable.')

print('\nTop-level output files:')
for path in sorted(Path('/kaggle/working').glob('rgca*')):
    print('-', path)


## 9. Real Image Retrieval Validation

This is the next milestone after the stress baseline. It hydrates only the JPG files referenced by the pilot subset, then runs BioMedCLIP retrieval-only validation. It does not download the full MIMIC-CXR-JPG archive.

Required Kaggle Secrets:

- `PHYSIONET_USERNAME`
- `PHYSIONET_PASS`


In [ ]:
RUN_BIOMEDCLIP_RETRIEVAL = True
HYDRATED_SUBSET_JSONL = Path('/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl')
BIOMEDCLIP_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0')

if RUN_BIOMEDCLIP_RETRIEVAL:
    if not PHYSIONET_USERNAME or not PHYSIONET_PASS:
        raise RuntimeError(
            'BioMedCLIP retrieval needs pilot JPGs. Add and enable Kaggle Secrets '
            'PHYSIONET_USERNAME and PHYSIONET_PASS, then rerun this cell.'
        )

    install_command = [sys.executable, '-m', 'pip', 'install', '-q', 'open_clip_torch', 'pillow']
    print('+', ' '.join(install_command))
    subprocess.run(install_command, check=True)

    hydrate_command = [
        sys.executable,
        'scripts/kaggle_hydrate_mimic_images.py',
        '--subset', str(SUBSET_JSONL),
        '--physionet-user', PHYSIONET_USERNAME,
        '--output-root', '/kaggle/working/physionet/mimic-cxr-jpg/files',
        '--updated-subset', str(HYDRATED_SUBSET_JSONL),
    ]
    safe_hydrate_command = [part if part != PHYSIONET_USERNAME else '<PHYSIONET_USERNAME>' for part in hydrate_command]
    print('+', ' '.join(safe_hydrate_command))
    subprocess.run(hydrate_command, check=True, env=os.environ.copy())

    validate_command = [
        sys.executable,
        'scripts/run_retrieval_validation.py',
        '--subset', str(HYDRATED_SUBSET_JSONL),
        '--backend', 'biomedclip',
        '--output-dir', str(BIOMEDCLIP_OUTPUT_DIR),
        '--top-k', '3',
        '--eval-limit', '20',
    ]
    print('+', ' '.join(validate_command))
    subprocess.run(validate_command, check=True)

    summary = json.loads((BIOMEDCLIP_OUTPUT_DIR / 'retrieval_validation_summary.json').read_text())
    print(json.dumps(summary, indent=2)[:4000])
else:
    print('Skipping BioMedCLIP retrieval validation because RUN_BIOMEDCLIP_RETRIEVAL=False')


## 10. Package All Evidence

This creates one downloadable archive containing the stress baseline outputs, BioMedCLIP retrieval validation outputs if run, manifests, summaries, and tables.


In [ ]:
ALL_EVIDENCE_ZIP = Path('/kaggle/working/rgca_research_evidence_v0.zip')
EXPERIMENT_ROOT = Path('/kaggle/working/rgca_experiments')

if EXPERIMENT_ROOT.exists():
    archive_base = ALL_EVIDENCE_ZIP.with_suffix('')
    shutil.make_archive(str(archive_base), 'zip', str(EXPERIMENT_ROOT))
    print('All evidence zipped for download:', ALL_EVIDENCE_ZIP)
    print('All evidence zip size MB:', round(ALL_EVIDENCE_ZIP.stat().st_size / (1024 * 1024), 3))
else:
    raise FileNotFoundError(f'Cannot create evidence zip because experiment root is missing: {EXPERIMENT_ROOT}')

if HYDRATED_SUBSET_JSONL.exists():
    print('Hydrated subset:', HYDRATED_SUBSET_JSONL)
    manifest = HYDRATED_SUBSET_JSONL.parent / 'image_hydration_manifest.json'
    print('Image hydration manifest:', manifest, 'exists=', manifest.exists())
